Title: PSL_anomaly_15d_clim_fut.ipynb

Purpose: Calculate a model mean 15 d climatology and the psl anomaly data for the future time period

Author: Onno Nennecke on 02.06.2025 Modified: 01.04.2026

Input data: 

- PSL Data from CMIP
    - These files lie here: /climca/people/onennecke/not_debiased_data_future/
- Used Runs File: CESM2_LE_runs.csv
    - These files lie here: /home/onennecke/CMIP_models/

Output data:



### Load libraries and functions

In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import os
import glob
import time
# import cftime
import re

# Importing functions
import Functions.grid_func as grid_func

/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
def setup_gridlines(ax, deg = 20, alpha = 0.4):
    gl = ax.gridlines(draw_labels=True, crs=ccrs.PlateCarree(), alpha=alpha)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 8}
    gl.ylabel_style = {'size': 8}
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlocator = MultipleLocator(deg)
    gl.ylocator = MultipleLocator(deg)

### Read data

In [3]:
# Read the dataframe from the csv file
used_runs_CMIP = pd.read_csv('/home/onennecke/CMIP_models/CMIP6_runs_future.csv')

used_runs_CESM = pd.read_csv('/home/onennecke/CMIP_models/CESM2_LE_runs.csv')

used_runs = pd.concat([used_runs_CMIP, used_runs_CESM], ignore_index=True)

used_runs

# Change the ref column to 1 for the first instance of each model
# used_runs['Ref'] = used_runs.groupby(['ESM', 'Institution']).cumcount().apply(lambda x: 1 if x == 0 else 0)

ESMs = used_runs['ESM'].unique()

In [4]:
used_runs

,ESM,Institution,run
0,ACCESS-CM2,CSIRO-ARCCSS,r4i1p1f1
1,ACCESS-CM2,CSIRO-ARCCSS,r5i1p1f1
2,ACCESS-CM2,CSIRO-ARCCSS,r1i1p1f1
3,BCC-CSM2-MR,BCC,r1i1p1f1
4,CESM2,NCAR,r4i1p1f1
...,...,...,...
187,CESM2,NCAR,LE2-1301_016
188,CESM2,NCAR,LE2-1301_017
189,CESM2,NCAR,LE2-1301_018
190,CESM2,NCAR,LE2-1301_019


In [5]:
base_path = '/climca/people/onennecke/not_debiased_data_future/'
psl_files = []
for i in range(len(used_runs)):
    ESM = used_runs.loc[i, 'ESM']
    run = used_runs.loc[i, 'run']
    
    # Construct the file path
    file_path = os.path.join(base_path, f'{ESM}_{run}_psl.nc')
    # Check if the file exists    
    if not os.path.isfile(file_path):
        print(f'File not found: {file_path}')
    else:
        psl_files.append(file_path)
print(len(psl_files))
# psl_files 
psl_ds = xr.open_dataset(psl_files[20])  # Open the first file to check the structure
psl_ds.load()

# Open the files using xarray
# psl_ds = xr.open_mfdataset(psl_files, combine='by_coords')



192


<xarray.Dataset> Size: 29MB
Dimensions:   (time: 3650, lat: 40, lon: 50)
Coordinates:
  * lat       (lat) int64 320B 30 31 32 33 34 35 36 37 ... 63 64 65 66 67 68 69
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
  * lon       (lon) int64 400B 340 341 342 343 344 345 346 ... 24 25 26 27 28 29
  * time      (time) datetime64[ns] 29kB 2037-01-01 2037-01-02 ... 2046-12-31
    ESM       <U9 36B 'EC-Earth3'
    run       <U8 32B 'r5i1p1f1'
    ESM_run   <U18 72B 'EC-Earth3_r5i1p1f1'
Data variables:
    psl       (time, lat, lon) float32 29MB 1.03e+05 1.03e+05 ... 9.815e+04
Attributes:
    regrid_method:  bilinear

In [7]:
out_dir = '/climca/people/onennecke/model_output/psl_anomaly/not_bc_fut/'

for esm in ESMs:
    ds_list = []
    esm_files = [fn for fn in psl_files if esm in os.path.basename(fn)]
    output_file = f'{esm}_psl_anomaly.nc'
    if os.path.isfile(os.path.join(out_dir, output_file)):
        print(f'{output_file} already exists. Skipping {esm}.')
        continue
    
    print(f'Processing {esm} with {len(esm_files)} files...')
    ds = xr.open_mfdataset(esm_files, combine='nested', concat_dim='ESM_run')
    ds_fixed = ds.convert_calendar('noleap', align_on='year')
    ds_mean = ds_fixed.mean('ESM_run')
    
    
    clim = ds_mean.groupby('time.dayofyear').mean('time')
    
    clim_pad = xr.concat([clim.isel(dayofyear=slice(-7, None)),
                          clim,
                          clim.isel(dayofyear=slice(0, 7))],
                         dim='dayofyear')
    clim_smooth = (
        clim_pad
        .rolling(dayofyear=15, center=True)
        .mean()
        .isel(dayofyear=slice(7, 7+365))
    )
    
    anom = ds_fixed.groupby('time.dayofyear') - clim_smooth

    anom = anom.sel(time=anom['time.month'].isin([10,11,12,1,2,3]))
    
    encoding = {}

    all_vars = {**anom.data_vars, **anom.coords}

    for var, da in all_vars.items():
        if da.dtype.kind in {"U", "S"}:
            values = da.values.astype(str)
            if values.ndim == 0:
                maxlen = len(str(values))
            else:
                maxlen = max(map(len, values.flatten()))
            encoding[var] = {"dtype": f"U{maxlen}"}
    encoding
    
    anom.to_netcdf(os.path.join(out_dir, output_file), encoding=encoding)
    
    # break

Processing ACCESS-CM2 with 3 files...
Processing BCC-CSM2-MR with 1 files...
Processing CESM2 with 103 files...


/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 103
  result = blockwise(


Processing EC-Earth3 with 54 files...


/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 54
  result = blockwise(


Processing GFDL-ESM4 with 1 files...
Processing KACE-1-0-G with 3 files...
Processing MPI-ESM1-2-HR with 10 files...


/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(


Processing MRI-ESM2-0 with 5 files...
Processing TaiESM1 with 1 files...
Processing UKESM1-0-LL with 11 files...


/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 11
  result = blockwise(
